### Structured Output

Models can be requested to provide their response in a format matching agreement schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing . Langchain supports multiple schema types and methods for enforcing structured output .

### Pydantic

Pydantic models provide the richest feature set with field validation, description, and nested structures

In [1]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
import os

load_dotenv()
# os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

model = ChatNVIDIA(
    model = "nvidia/nemotron-3-ultra-550b-a55b",
    api_key = os.getenv("NVIDIA_API_KEY")
)
model

ChatNVIDIA(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-nvidia-ai-endpoints': '1.4.3'}}, profile={}, base_url='https://integrate.api.nvidia.com/v1', model='nvidia/nemotron-3-ultra-550b-a55b', default_headers={}, model_kwargs={})

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie rating out of 10")

In [7]:
model_structured_output = model.with_structured_output(Movie)
model_structured_output

<langchain_nvidia_ai_endpoints.chat_models.ChatNVIDIA.with_structured_output.<locals>._FallbackRunnable at 0x275bcfc4050>

In [9]:
model.invoke("Please provide the details of the movie 'Inception'")

AIMessage(content='Here is a comprehensive breakdown of the movie **Inception (2010)**.\n\n---\n\n### **Quick Facts**\n*   **Title:** Inception\n*   **Release Date:** July 16, 2010 (Premiere in London); July 16, 2010 (US)\n*   **Director:** Christopher Nolan\n*   **Writer:** Christopher Nolan\n*   **Producer:** Christopher Nolan, Emma Thomas\n*   **Cinematography:** Wally Pfister\n*   **Music:** Hans Zimmer\n*   **Editor:** Lee Smith\n*   **Production Companies:** Syncopy Inc., Legendary Pictures\n*   **Distributor:** Warner Bros. Pictures\n*   **Budget:** $160 million\n*   **Box Office:** $839 million (worldwide)\n*   **Runtime:** 148 minutes\n*   **Rating:** PG-13 (for sequences of violence and action throughout)\n\n---\n\n### **Principal Cast**\n| Actor | Character | Role Description |\n| :--- | :--- | :--- |\n| **Leonardo DiCaprio** | **Dom Cobb** | A skilled "Extractor" who steals secrets from the subconscious. Haunted by the death of his wife. |\n| **Marion Cotillard** | **Mal Co

In [10]:
model_structured_output.invoke("Please provide the details of the movie 'Inception'")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [6]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [ ]:
'''ChatNVIDIA has not implemented include_raw option'''
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie rating out of 10")

model_structured_output = model.with_structured_output(Movie, include_raw=True)
model_structured_output

{
  raw: _ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.4'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x00000275BD8927B0

### Nested Structure

In [3]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    home : str

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    cast: list[Actor]
    genres: list[str] = Field(description="The director of the movie")
    budget: float | None = Field(None, description=" Budget in millions usd")
    
model_structured_output = model.with_structured_output(Movie)
response = model_structured_output.invoke("Please provide the details of the movie 'Inception'")

response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Movie(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', home='Los Angeles'), Actor(name='Joseph Gordon-Levitt', home='Los Angeles'), Actor(name='Elliot Page', home='Halifax'), Actor(name='Tom Hardy', home='London')], genres=['Science Fiction', 'Action', 'Adventure'], budget=160.0)

### TypedDict

TypeDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation .

In [ ]:
'''ChatNVIDIA ddoesn't support'''
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]

TypedDict_output = model.with_structured_output(Movie)
response = TypedDict_output.invoke("Please provide the details of the movie 'Avenger'")

response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [4]:
from typing_extensions import TypedDict

class Actor(TypedDict):
    name : str
    home : str

class Movie(TypedDict):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    cast: list[Actor]
    genres: list[str] = Field(description="The director of the movie")
    budget: float | None = Field(None, description=" Budget in millions usd")
    
model_structured_output = model.with_structured_output(Movie)
response = model_structured_output.invoke("Please provide the details of the movie 'Avengers'")

response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'home': 'United States'},
  {'name': 'Chris Evans', 'home': 'United States'},
  {'name': 'Mark Ruffalo', 'home': 'United States'},
  {'name': 'Chris Hemsworth', 'home': 'Australia'},
  {'name': 'Scarlett Johansson', 'home': 'United States'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [8]:
model.profile

{'name': 'Gemini 3.1 Flash Lite',
 'release_date': '2026-05-07',
 'last_updated': '2026-05-07',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

### DataClasses
A data class is a class type will clearly containing mainly data although, there are not really any restrictions. you created using @dataclass decorator

In [ ]:
# import os
# os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

In [ ]:
# import os
# from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


# Explicitly initialize the Gemini Developer API
'''culprit vertexai wtf is this'''
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False, 
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Extract contact info from: "
                "John Doe, john@example.com, (555) 123-4567"
            )
        }
    ]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9f15ce79-cd1a-4642-9655-9c9796285bb7'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', 'extras': {'signature': 'EnMKcQFpFH0TbIg/N2iQrP7szCpu1w5vdu9Is4+xFdgn9k8tj5Ez4lA2jJAkiBL4F3fUsqPpOMbs7roBZUVJX1hw4QeUEZw0qJQi/QmrKHsFUS1wcAu0bKvSO6qSkiZQIenkqMVQ7TICWbTSsN99CNJJxBSa'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14ce5-7696-7e43-88aa-acbca5401cf5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [13]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [14]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str
    email: str
    phone: str


# Explicitly initialize the Gemini Developer API
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": ("Extract contact info from: John Doe, john@example.com, (555) 123-4567")}]
})

print(result["structured_response"])


{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


#### DataClass
using you can do the same thing

In [3]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

import os
from dotenv import load_dotenv

load_dotenv()

@dataclass
class ContactInfo:
    ''' Contact information of a person'''
    name : str
    email : str
    phone : str

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": ("Extract contact info from: John Doe, john@example.com, (555) 123-4567")}]
})

print(result["structured_response"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
